# Lecture 6: Text Splitters & Chunking Strategies

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Intermediate  

---

## The Big Picture

In Lecture 5, we learned how to **load** documents. But here's the problem:

> A 100-page PDF becomes one giant wall of text.  
> An LLM can't process that. A search engine can't match against that.  
> We need to **cut it into smart, bite-sized pieces** — that's chunking.

**Chunking is one of the most important decisions in any RAG system.**  
Bad chunks = bad retrieval = bad answers, no matter how good your LLM is.

### What You Will Learn

| # | Topic | Real-World Analogy |
|---|-------|--------------------|
| 1 | Why we chunk documents | Why you slice a pizza before serving |
| 2 | Chunk size — the Goldilocks problem | Slices too small or too big |
| 3 | Overlap — why chunks share text | Pages in a book that repeat the last sentence |
| 4 | RecursiveCharacterTextSplitter | The smart pizza cutter |
| 5 | Other splitters | Specialized tools for special jobs |
| 6 | Metadata preservation | Never lose track of where a chunk came from |
| 7 | Hands-on: split, compare, choose | Build it yourself! |

> **Key Insight:** Chunking is where most RAG pipelines silently fail.  
> Master this, and you're ahead of 90% of beginners.

---

## 0. Environment Setup

Run this cell **once** to install the packages we'll need today.  
If you already have them from Lecture 5, you're good to go!

In [3]:
# Install required packages (run once, then you can skip this cell)
%pip install langchain langchain-community langchain-text-splitters pypdf

Note: you may need to restart the kernel to use updated packages.


---

## 1. Why Do We Chunk Documents?

### The Pizza Analogy

Imagine you ordered a pizza. The chef hands you the **entire uncut pizza**.  
You can't eat it like that! You need to cut it into slices.

But HOW you cut it matters:
- **Too small** (tiny squares) — each piece has barely any topping, hard to enjoy
- **Too big** (half the pizza) — too much to handle at once
- **Just right** (proper slices) — each slice is a satisfying, complete portion

Documents work the same way. Here are the 3 reasons we chunk:

### Reason 1: Context Window Limits
LLMs can only read a limited amount of text at once (the "context window").  
You can't send 500 pages to an LLM — it will either crash or ignore most of it.

### Reason 2: Retrieval Precision
When a user asks a question, you want to find the **exact 2-3 paragraphs** that  
answer it — not dump 50 pages and hope the LLM figures it out.

### Reason 3: Semantic Coherence
Each chunk should contain **one complete idea**. If you cut in the middle of a  
sentence, the chunk becomes meaningless.

> **Bottom line:** Bad chunking = bad RAG. No matter how good your LLM is,  
> if you feed it garbage chunks, you get garbage answers.

In [4]:
# Let's SEE why chunking matters with a real example
# First, load our sample article from Lecture 5's data folder

from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path="data/nlp_article.txt", encoding="utf-8")
documents = loader.load()

# How big is this document?
# documents[0] gets the first (and only) document from the list
full_text = documents[0].page_content

# len() counts the total number of characters in the string
print(f"Document length: {len(full_text):,} characters")

# Rough token estimate: 1 token ~ 4 characters in English
# // is integer division (divides and rounds down, no decimals)
print(f"That's roughly {len(full_text) // 4:,} tokens")

# [:300] shows only the first 300 characters as a preview
# To see the FULL document, use: print(full_text)
print(f"\nFirst 300 characters:")
print(full_text[:300])

Document length: 8,039 characters
That's roughly 2,009 tokens

First 300 characters:
Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objectiv


#### What just happened?

We loaded a ~8,000 character article. That's roughly 2,000 tokens.  
This is small enough for most LLMs, but imagine a 200-page PDF —  
that could be 500,000+ characters. Way too much to send at once!

**Quick math:** 1 token ~ 4 characters (rough estimate for English).  
GPT-4's context window is ~128K tokens. Claude's is ~200K tokens.  
But just because it *fits* doesn't mean it's *effective* — smaller,  
focused chunks always give better retrieval results.

---

## 2. Chunk Size — The Goldilocks Problem

Choosing the right chunk size is like Goldilocks and the three bears:  
not too small, not too big, but just right.

### What Happens at Different Sizes?

| | Too Small (&lt;200 chars) | Just Right (500-1500 chars) | Too Large (&gt;2000 chars) |
|---|---|---|---|
| **Example chunk** | `"NLP is a branch of..."` | `"NLP is a branch of AI that helps computers understand language..."` | `"NLP is a branch of AI that... [3 pages of text] ...too much noise!"` |
| **Problem** | Lost context! Can't understand the idea. | One complete idea per chunk. | Too much noise in search results. |
| **Analogy** | Pizza cut into tiny squares — no topping per piece | Pizza cut into proper slices — satisfying portions | Half the pizza on one plate — too much to handle |

### The Sweet Spot: 500-1500 Characters

| Chunk Size | Characters | Good For |
|-----------|------------|----------|
| Small | 200-500 | FAQ answers, short paragraphs |
| Medium | 500-1000 | General purpose (good default) |
| Large | 1000-1500 | Technical docs, detailed explanations |
| Very Large | 1500-2000 | Long-form articles, legal documents |

There's no single "correct" size — it depends on your data and use case.  
We'll experiment with different sizes later in this notebook!

In [5]:
# Let's see what DIFFERENT chunk sizes look like on real text
# We'll manually slice the text to build intuition BEFORE using LangChain

# [:2000] takes only the first 2000 characters of our article for this demo
# To use the full article, replace with: sample_text = full_text
sample_text = full_text[:2000]

# We'll test 3 different chunk sizes to see how they compare
chunk_sizes = [100, 500, 1000]

# This loop runs 3 times — once for each size in the list
for size in chunk_sizes:
    # // is integer division: 2000 // 500 = 4 chunks
    num_chunks = len(sample_text) // size

    # [:size] slices the text from the start to the chunk size
    # If size=100, first_chunk = first 100 characters
    # If size=500, first_chunk = first 500 characters
    first_chunk = sample_text[:size]

    print(f"\n{'=' * 60}")
    print(f"CHUNK SIZE: {size} characters")
    print(f"Number of chunks from 2000 chars: {num_chunks}")
    print(f"First chunk preview:")
    print(f"  '{first_chunk}'")

    # [size-20:size] slices the LAST 20 characters of the first chunk
    # This shows you where exactly the chunk gets cut off
    print(f"  --- ends at: '...{sample_text[size - 20:size]}'")


CHUNK SIZE: 100 characters
Number of chunks from 2000 chars: 20
First chunk preview:
  'Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?
'
  --- ends at: '...anguage Processing?
'

CHUNK SIZE: 500 characters
Number of chunks from 2000 chars: 4
First chunk preview:
  'Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.

NLP combines computational linguistics — rule-based modeling of human lan'
  --- ends at: '...odeling of human lan'

CHUNK SIZE: 1000 characters
Number of chunks from 2000 chars: 2
First chunk preview:
  'Natural Language Processing: A Comprehensive Gui

#### What just happened?

Look at where each chunk **ends**:

- **100 chars** — probably cuts in the middle of a sentence! The chunk makes no sense on its own.
- **500 chars** — might cut mid-paragraph, but at least has some context.
- **1000 chars** — likely contains a full paragraph or two. Much more useful.

**The problem with simple slicing:** It cuts at exact character counts, even if  
that's in the middle of a word! That's why we need smart splitters (coming next).

---

## 3. Overlap — Why Chunks Should Share Some Text

### The Problem Without Overlap

Imagine this sentence sits right at the boundary between two chunks:

```
Chunk 1: "...BERT was released by Google."
Chunk 2: "It achieved state-of-the-art results on 11 NLP tasks."
```

If someone asks *"What did BERT achieve?"*, neither chunk alone has the full answer!  
Chunk 1 knows about BERT but not the achievements. Chunk 2 has the achievements  
but "It" doesn't tell us what "it" refers to.

### The Solution: Overlap

Overlap means the **end of one chunk is repeated at the start of the next chunk**:

```
Chunk 1: "...BERT was released by Google. It achieved state-of-the-art"
                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                  This part appears in BOTH chunks (overlap)
Chunk 2: "BERT was released by Google. It achieved state-of-the-art results on 11..."
```

Now BOTH chunks have the complete context!

### How Much Overlap?

**Rule of thumb: 10-20% of your chunk size.**

| Chunk Size | Recommended Overlap | Why |
|-----------|--------------------|----- |
| 500 chars | 50-100 chars | ~1-2 sentences of context |
| 1000 chars | 100-200 chars | ~2-3 sentences of context |
| 1500 chars | 150-300 chars | ~3-5 sentences of context |

**Too much overlap** (>30%) = wastes storage and duplicates too much info  
**Too little overlap** (0%) = important context gets lost at boundaries

In [6]:
# Let's visualize HOW overlap works with a simple example

demo_text = (
    "BERT was released by Google in 2018. "
    "It achieved state-of-the-art results on 11 NLP tasks. "
    "GPT was developed by OpenAI. "
    "It uses a different training approach called autoregressive modeling."
)

chunk_size = 80
overlap = 30

print(f"Full text ({len(demo_text)} chars): '{demo_text}'")
print(f"\nChunk size: {chunk_size} | Overlap: {overlap}")
print("=" * 60)

# step = how many characters we move forward before starting the next chunk
# If chunk_size=80 and overlap=30, step=50
# This means we skip 50 new chars, then repeat the last 30 from the previous chunk
step = chunk_size - overlap
chunk_number = 1
start = 0

# This while loop keeps creating chunks until we've covered the entire text
# start tracks our current position in the text
while start < len(demo_text):
    end = start + chunk_size

    # [start:end] slices out one chunk from the text
    # e.g., demo_text[0:80] gives chars 0-79, demo_text[50:130] gives chars 50-129
    chunk = demo_text[start:end]

    # min(end, len(demo_text)) ensures we don't show an index beyond the text length
    print(f"\nChunk {chunk_number} (chars {start}-{min(end, len(demo_text))}):")
    print(f"  '{chunk}'")

    chunk_number += 1
    start += step  # move forward by 50, NOT 80 — that's what creates the overlap

Full text (189 chars): 'BERT was released by Google in 2018. It achieved state-of-the-art results on 11 NLP tasks. GPT was developed by OpenAI. It uses a different training approach called autoregressive modeling.'

Chunk size: 80 | Overlap: 30

Chunk 1 (chars 0-80):
  'BERT was released by Google in 2018. It achieved state-of-the-art results on 11 '

Chunk 2 (chars 50-130):
  'tate-of-the-art results on 11 NLP tasks. GPT was developed by OpenAI. It uses a '

Chunk 3 (chars 100-180):
  'eveloped by OpenAI. It uses a different training approach called autoregressive '

Chunk 4 (chars 150-189):
  'pproach called autoregressive modeling.'


#### What just happened?

- Each chunk is 80 characters long, but we only move forward by **50** characters  
  (80 - 30 = 50) before starting the next chunk
- This means the last 30 characters of each chunk appear again at the start  
  of the next chunk — that's the **overlap**
- Look at the chunks: you should see repeated text between consecutive chunks

**Why `step = chunk_size - overlap`?** If chunks are 80 chars with 30 overlap,  
we slide the window forward by only 50 chars. The remaining 30 chars are shared  
with the next chunk. This is exactly how LangChain's splitters work internally!

---

## 4. RecursiveCharacterTextSplitter — The Smart Pizza Cutter

This is the **#1 most commonly used splitter** in LangChain, and your best default.

### Why "Recursive"?

It tries to split text at **natural boundaries**, working through a priority list:

```
Priority 1: Split at paragraph breaks  ("\n\n")
     ↓ (if chunks still too big)
Priority 2: Split at line breaks       ("\n")
     ↓ (if chunks still too big)
Priority 3: Split at spaces            (" ")
     ↓ (if chunks still too big)
Priority 4: Split at any character     ("")
```

**Analogy:** Imagine you're asked to divide a chapter into smaller sections.  
You'd first try to split between paragraphs. If a paragraph is still too long,  
you'd split between sentences. Only as a last resort would you split mid-word.

That's exactly what `RecursiveCharacterTextSplitter` does!

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the splitter with our chosen settings
# chunk_size = maximum number of characters per chunk
# chunk_overlap = how many characters to repeat between chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # each chunk will be at most 1000 characters
    chunk_overlap=200,   # 200 chars of overlap (20% — right in the sweet spot)
)

# Split our loaded document into chunks
# .split_documents() takes a list of Documents and returns a list of smaller Documents
chunks = splitter.split_documents(documents)

print(f"Original document: 1 document, {len(full_text):,} characters")
print(f"After splitting: {len(chunks)} chunks")
print(f"\nSettings: chunk_size=1000, chunk_overlap=200")

Original document: 1 document, 8,039 characters
After splitting: 10 chunks

Settings: chunk_size=1000, chunk_overlap=200


In [8]:
# Let's inspect the first 3 chunks in detail

# chunks[:3] slices the list to get only the first 3 chunks
# To see ALL chunks, remove [:3] → for i, chunk in enumerate(chunks):
# enumerate() gives us the index (i) and the chunk object in each iteration
for i, chunk in enumerate(chunks[:3]):
    print(f"\n{'=' * 60}")
    print(f"CHUNK {i + 1} of {len(chunks)}")
    print(f"{'=' * 60}")
    print(f"Length: {len(chunk.page_content)} characters")
    print(f"Metadata: {chunk.metadata}")

    # Here we print the FULL chunk content (no slicing!)
    # Since chunks are already small (~1000 chars), it's safe to show everything
    print(f"\nContent:")
    print(chunk.page_content)
    print(f"\n--- End of chunk {i + 1} ---")


CHUNK 1 of 10
Length: 773 characters
Metadata: {'source': 'data/nlp_article.txt'}

Content:
Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.

NLP combines computational linguistics — rule-based modeling of human language — with statistical, machine learning, and deep learning models. Together, these technologies enable computers to process human language in the form of text or voice data and to understand its full meaning, complete with the speaker's or writer's intent and sentiment.

--- End of chunk 1 ---

CHUNK 2 of 10
Length: 834 characters
Metadata: {'source': 'data/nlp_article.txt'}

Content:
The history of N

In [9]:
# Now let's verify the overlap — does the end of chunk 1 appear at the start of chunk 2?

if len(chunks) >= 2:
    chunk_1_text = chunks[0].page_content
    chunk_2_text = chunks[1].page_content

    print("OVERLAP VERIFICATION")
    print("=" * 60)

    # IMPORTANT: The overlap may NOT be exactly 200 characters!
    # RecursiveCharacterTextSplitter splits at natural boundaries (paragraphs,
    # sentences). It TRIES to keep ~200 chars of overlap, but adjusts to avoid
    # cutting mid-sentence. So the real overlap could be 150, 180, or even 0
    # if a clean paragraph break happens to fall right at the boundary.

    # To find the ACTUAL overlap, we check: does the START of chunk 2
    # appear somewhere near the END of chunk 1?
    # We search for increasingly smaller portions of chunk 2's start in chunk 1.
    actual_overlap = 0
    for size in range(min(300, len(chunk_2_text)), 0, -1):
        # chunk_2_text[:size] takes the first 'size' characters of chunk 2
        # We check if that text appears at the END of chunk 1
        if chunk_1_text.endswith(chunk_2_text[:size]):
            actual_overlap = size
            break  # stop at the first (largest) match

    print(f"\nChunk 1 length: {len(chunk_1_text)} chars")
    print(f"Chunk 2 length: {len(chunk_2_text)} chars")
    print(f"Requested overlap: 200 chars")
    print(f"Actual overlap found: {actual_overlap} chars")

    if actual_overlap > 0:
        # Show the overlapping text that appears in BOTH chunks
        print(f"\n--- Shared text (appears in both chunks) ---")
        print(f"'{chunk_2_text[:actual_overlap]}'")
    else:
        # This happens when the splitter found a clean paragraph break
        # right at the boundary — no overlap was needed
        print(f"\nNo overlap found! This means the splitter found a clean")
        print(f"paragraph break between these chunks. The overlap setting")
        print(f"is a MAXIMUM — the splitter won't force overlap if there's")
        print(f"a natural break point.")
        print(f"\n--- End of Chunk 1 (last 100 chars) ---")
        print(f"'{chunk_1_text[-100:]}'")
        print(f"\n--- Start of Chunk 2 (first 100 chars) ---")
        print(f"'{chunk_2_text[:100]}'")

OVERLAP VERIFICATION

Chunk 1 length: 773 chars
Chunk 2 length: 834 chars
Requested overlap: 200 chars
Actual overlap found: 0 chars

No overlap found! This means the splitter found a clean
paragraph break between these chunks. The overlap setting
is a MAXIMUM — the splitter won't force overlap if there's
a natural break point.

--- End of Chunk 1 (last 100 chars) ---
'ta and to understand its full meaning, complete with the speaker's or writer's intent and sentiment.'

--- Start of Chunk 2 (first 100 chars) ---
'The history of NLP dates back to the 1950s when Alan Turing published his famous article "Computing '


#### What just happened?

1. We created a `RecursiveCharacterTextSplitter` with `chunk_size=1000` and `chunk_overlap=200`
2. It split our single document into multiple smaller chunks
3. Each chunk is roughly 800-1000 characters (the splitter tries to stay under the limit)

**Important nuance about overlap:**

`chunk_overlap=200` is a **maximum**, not a guarantee. The splitter prioritizes
clean breaks (paragraph → sentence → word), so:

- If a paragraph break falls right at the boundary → overlap might be **0**
  (no need to repeat text — both chunks start/end at a clean break)
- If the break happens mid-paragraph → overlap will be close to **200 chars**
- The overlap is most useful when text flows continuously without clear breaks

**Think of it this way:** Overlap is a safety net for when the splitter *has to*
cut in the middle of flowing text. When it finds a natural break point, the
safety net isn't needed.

**The pattern is simple:**  
`splitter = RecursiveCharacterTextSplitter(chunk_size=N, chunk_overlap=M)`  
`chunks = splitter.split_documents(docs)`

---

## 5. Other Splitter Types — Specialized Tools for Special Jobs

**`RecursiveCharacterTextSplitter` is your default.** But just like Lecture 5  
showed us that different PDFs need different loaders, different *content types*  
need different splitters.

| Splitter | Best For | How It Splits |
|----------|----------|---------------|
| `RecursiveCharacterTextSplitter` | General text (your default) | Paragraphs → lines → spaces → chars |
| `CharacterTextSplitter` | Simple, fast splitting | Splits on a single separator (e.g., `\n\n`) |
| `TokenTextSplitter` | When you need exact token counts | Splits by tokens, not characters |
| `MarkdownTextSplitter` | Markdown docs, wikis | Splits on `#` headers — preserves structure |
| `RecursiveCharacterTextSplitter.from_language()` | Source code | Splits at function/class boundaries |
| `HTMLSectionSplitter` | HTML pages | Splits on HTML tags like `<h1>`, `<h2>` |

In [10]:
# ============================================================
# CharacterTextSplitter — The Simple Cutter
# ============================================================
# Splits on a SINGLE separator. Fast, but less smart than Recursive.
# Good for: data that already has clear separators (like logs or CSV-like text)

from langchain_text_splitters import CharacterTextSplitter

# Split only on double newlines (paragraph breaks)
simple_splitter = CharacterTextSplitter(
    separator="\n\n",    # only split at paragraph breaks — won't split mid-paragraph
    chunk_size=1000,
    chunk_overlap=200,
)

simple_chunks = simple_splitter.split_documents(documents)

print(f"CharacterTextSplitter produced: {len(simple_chunks)} chunks")

# [:300] shows a preview of the first chunk (first 300 characters only)
# To see the full first chunk, use: print(simple_chunks[0].page_content)
print(f"\nFirst chunk ({len(simple_chunks[0].page_content)} chars):")
print(simple_chunks[0].page_content[:300])

CharacterTextSplitter produced: 10 chunks

First chunk (773 chars):
Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objectiv


#### What just happened?

`CharacterTextSplitter` only looks for ONE separator (`\n\n` = paragraph break).  
If a paragraph is bigger than `chunk_size`, it gets forced through as-is  
(unless you set `is_separator_regex=True` for more control).

**When to use this instead of Recursive?**
- When your text has very **consistent formatting** (e.g., one paragraph per topic)
- When speed matters more than perfect splitting
- When you want full control over the exact split point

In [11]:
# ============================================================
# TokenTextSplitter — Split by Tokens, Not Characters
# ============================================================
# Why tokens matter: LLM APIs charge per TOKEN, not per character.
# 1 token ~ 4 characters in English, but this varies by language.
# This splitter ensures each chunk has EXACTLY the right number of tokens.
# %pip install tiktoken
from langchain_text_splitters import TokenTextSplitter

# Split into chunks of 256 tokens with 50 tokens of overlap
# NOTE: these are TOKEN counts, not character counts!
token_splitter = TokenTextSplitter(
    chunk_size=256,      # 256 tokens per chunk (not characters!)
    chunk_overlap=50,    # 50 tokens of overlap
)

token_chunks = token_splitter.split_documents(documents)

print(f"TokenTextSplitter produced: {len(token_chunks)} chunks")
print(f"\nFirst chunk:")
print(f"  Characters: {len(token_chunks[0].page_content)}")

# [:200] shows a preview — the first 200 characters of the chunk
# To see the full chunk, use: print(token_chunks[0].page_content)
print(f"  Content preview: {token_chunks[0].page_content[:200]}...")

TokenTextSplitter produced: 8 chunks

First chunk:
  Characters: 1320
  Content preview: Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that foc...


#### What just happened?

- `TokenTextSplitter` counts **tokens** instead of characters
- This is important when you need to stay within an LLM's token limit
- Notice the character count varies between chunks — that's because different  
  words use different numbers of tokens ("a" = 1 token, "unbelievable" = 3 tokens)

**When to use this?**
- When you're sending chunks directly to an LLM and need exact token control
- When you're calculating API costs and need predictable chunk sizes
- When working with non-English text (where char-to-token ratio varies a lot)

In [12]:
# ============================================================
# MarkdownTextSplitter — For Markdown and Wiki Content
# ============================================================
# Splits on markdown headers (#, ##, ###) so each section stays together.
# Perfect for: documentation, wikis, README files, knowledge bases.

from langchain_text_splitters import MarkdownTextSplitter

# Load a markdown file to demonstrate
md_loader = TextLoader(file_path="data/sample_docs.md", encoding="utf-8")
md_documents = md_loader.load()

# :, adds commas to the number (e.g., 1,234 instead of 1234)
print(f"Original markdown: {len(md_documents[0].page_content):,} characters")
print()

# Split on markdown structure
md_splitter = MarkdownTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

md_chunks = md_splitter.split_documents(md_documents)

print(f"Markdown chunks created: {len(md_chunks)}")

# This loop goes through EVERY markdown chunk
# enumerate() pairs each chunk with its index (0, 1, 2, ...)
for i, chunk in enumerate(md_chunks):
    # .split("\n") breaks the content into lines, [0] takes the first line
    # In markdown files, the first line is usually the section header
    first_line = chunk.page_content.split("\n")[0]
    print(f"\n  Chunk {i + 1} ({len(chunk.page_content)} chars):")
    print(f"  Starts with: '{first_line}'")

    # [:150] shows a preview of each chunk — first 150 characters
    # To see the FULL chunk, replace with: print(f"  {chunk.page_content}")
    print(f"  Preview: {chunk.page_content[:150]}...")

Original markdown: 1,130 characters

Markdown chunks created: 3

  Chunk 1 (490 chars):
  Starts with: '# Company Return Policy'
  Preview: # Company Return Policy

## Eligibility

All items purchased from our store can be returned within 30 days of the original purchase date. Items must b...

  Chunk 2 (460 chars):
  Starts with: '## Refund Processing'
  Preview: ## Refund Processing

Refunds are processed within 5-7 business days after we receive the returned item. The refund will be credited to the original p...

  Chunk 3 (195 chars):
  Starts with: '## Contact Support'
  Preview: ## Contact Support

If you have questions about your return, contact our support team at support@example.com or call 1-800-555-0123. Our support hours...


#### What just happened?

- `MarkdownTextSplitter` respects the **heading structure** of the document
- Each chunk tends to be one complete section (from one `##` header to the next)
- This is much better than blindly cutting every 500 characters!

**When to use this?**
- Documentation sites, README files, wikis
- Any content with clear heading-based structure
- Knowledge bases where each section covers a different topic

In [13]:
# ============================================================
# Code Splitter — For Source Code
# ============================================================
# Splits code at function/class boundaries instead of mid-function.
# Supports: Python, JavaScript, Go, Rust, Java, C++, and many more.

from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

# Example: split Python code (triple quotes define a multi-line string)
python_code = '''
def load_documents(file_path):
    """Load documents from a file path."""
    loader = TextLoader(file_path)
    documents = loader.load()
    return documents


def split_documents(documents, chunk_size=1000):
    """Split documents into smaller chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200,
    )
    chunks = splitter.split_documents(documents)
    return chunks


class DocumentProcessor:
    """Process documents through the full pipeline."""

    def __init__(self, chunk_size=1000):
        self.chunk_size = chunk_size

    def process(self, file_path):
        docs = load_documents(file_path)
        chunks = split_documents(docs, self.chunk_size)
        return chunks
'''

# .from_language() creates a splitter that understands a specific programming language
# It knows where functions and classes start/end in Python
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=200,      # small size to force splits for demo purposes
    chunk_overlap=30,
)

# .split_text() takes a plain string (not a Document object)
# Use .split_documents() when you have Document objects from a loader
code_chunks = python_splitter.split_text(python_code)

print(f"Python code split into: {len(code_chunks)} chunks")

# This loop prints every code chunk with its full content
# Since code chunks are small, we show everything (no slicing needed)
for i, chunk in enumerate(code_chunks):
    print(f"\n--- Code Chunk {i + 1} ({len(chunk)} chars) ---")
    print(chunk)

Python code split into: 5 chunks

--- Code Chunk 1 (159 chars) ---
def load_documents(file_path):
    """Load documents from a file path."""
    loader = TextLoader(file_path)
    documents = loader.load()
    return documents

--- Code Chunk 2 (173 chars) ---
def split_documents(documents, chunk_size=1000):
    """Split documents into smaller chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,

--- Code Chunk 3 (91 chars) ---
chunk_overlap=200,
    )
    chunks = splitter.split_documents(documents)
    return chunks

--- Code Chunk 4 (158 chars) ---
class DocumentProcessor:
    """Process documents through the full pipeline."""

    def __init__(self, chunk_size=1000):
        self.chunk_size = chunk_size

--- Code Chunk 5 (148 chars) ---
def process(self, file_path):
        docs = load_documents(file_path)
        chunks = split_documents(docs, self.chunk_size)
        return chunks


#### What just happened?

- `from_language(Language.PYTHON)` creates a splitter that understands Python syntax
- It splits at **class and function boundaries** instead of mid-function
- Notice `.split_text()` takes a string, while `.split_documents()` takes Document objects

**Supported languages:** Python, JavaScript, TypeScript, Java, Go, Rust, C, C++,  
Ruby, Scala, Swift, Markdown, LaTeX, HTML, and more!

### How to Choose the Right Splitter

```
What kind of content are you splitting?
  |
  ├─ General text (articles, books, reports)
  |    └─ RecursiveCharacterTextSplitter (your default)
  |
  ├─ Markdown or wiki pages
  |    └─ MarkdownTextSplitter
  |
  ├─ Source code
  |    └─ RecursiveCharacterTextSplitter.from_language()
  |
  ├─ Need exact token counts (for API cost control)
  |    └─ TokenTextSplitter
  |
  ├─ HTML pages
  |    └─ HTMLSectionSplitter
  |
  └─ Simple text with clear separators
       └─ CharacterTextSplitter
```

> **Same lesson as Lecture 5:** Don't memorize — learn to pick the right tool  
> for the job. Start with `RecursiveCharacterTextSplitter` and only switch  
> when your content has special structure.

---

## 6. Metadata Preservation — Never Lose Track of Your Chunks

When you split a document into chunks, each chunk **inherits the metadata**  
from the original document. This is crucial for citations!

**Imagine your RAG system answers:** *"The return policy allows 30-day returns."*  
Without metadata, you can't tell the user *where* that answer came from.  
With metadata, you can say: *"From page 5 of policy.pdf, paragraph 3."*

### What Metadata Is Preserved?

| Source | Metadata in Each Chunk |
|--------|------------------------|
| PDF | `source` (file), `page` (page number) |
| Web | `source` (URL), `title` |
| CSV | `source` (file), `row` (row number) |
| Text | `source` (file path) |

In [23]:
# Let's see metadata preservation in action

# First, load a PDF (each page has page number in metadata)
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader(file_path="data/sample.pdf")
pdf_docs = pdf_loader.load()

print(f"Loaded {len(pdf_docs)} pages from PDF")
print(f"Page 1 metadata: {pdf_docs[0].metadata}")

# Now split those pages into smaller chunks
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
pdf_chunks = pdf_splitter.split_documents(pdf_docs)

print(f"\nAfter splitting: {len(pdf_chunks)} chunks")
print("\n--- Metadata in each chunk (notice page numbers are preserved!) ---")

# [:5] shows the first 5 chunks only
# To see ALL chunks, remove [:5] → for i, chunk in enumerate(pdf_chunks):
for i, chunk in enumerate(pdf_chunks[:5]):
    # .get() safely reads a value from the metadata dictionary
    # If "page" doesn't exist, it returns "unknown" instead of crashing
    page = chunk.metadata.get("page", "unknown")
    source = chunk.metadata.get("source", "unknown")
    print(f"  Chunk {i + 1}: page={page}, source='{source}', length={len(chunk.page_content)} chars")

Loaded 3 pages from PDF
Page 1 metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-26T21:47:30+05:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'data/sample.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}

After splitting: 3 chunks

--- Metadata in each chunk (notice page numbers are preserved!) ---
  Chunk 1: page=0, source='data/sample.pdf', length=411 chars
  Chunk 2: page=1, source='data/sample.pdf', length=259 chars
  Chunk 3: page=2, source='data/sample.pdf', length=320 chars


In [24]:
# You can also ADD custom metadata to chunks after splitting
# This is useful for tracking chunk position, section names, etc.

# This loop goes through EVERY chunk and adds extra metadata fields
# enumerate() gives us the index (0, 1, 2, ...) as "index" and the chunk object
for index, chunk in enumerate(pdf_chunks):
    # We're adding new keys to the existing metadata dictionary
    # chunk.metadata is a dict, and you can add keys like: dict["new_key"] = value
    chunk.metadata["chunk_index"] = index
    chunk.metadata["total_chunks"] = len(pdf_chunks)

# Verify the enriched metadata
# [:3] shows only the first 3 chunks — to see all, remove the slice
print("Enriched metadata for first 3 chunks:")
for chunk in pdf_chunks[:3]:
    print(f"  {chunk.metadata}")

print(f"\nNow each chunk knows:")
print(f"  - Where it came from (source, page)")
print(f"  - Its position in the sequence (chunk_index)")
print(f"  - How many total chunks exist (total_chunks)")

Enriched metadata for first 3 chunks:
  {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-26T21:47:30+05:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'data/sample.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'chunk_index': 0, 'total_chunks': 3}
  {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-26T21:47:30+05:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'data/sample.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2', 'chunk_index': 1, 'total_chunks': 3}
  {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-26T21:47:

#### What just happened?

1. We loaded a 3-page PDF — each page had `source` and `page` in its metadata
2. After splitting, each chunk **inherited** the metadata from its parent page
3. We then **enriched** the metadata by adding `chunk_index` and `total_chunks`

**Why add custom metadata?**
- `chunk_index` lets you reconstruct the original order
- You could add `section_title`, `timestamp`, `author` — anything useful
- This metadata travels with the chunk into the vector store and back to the user

**Golden rule:** Never lose track of where a chunk came from.  
Your users will want citations, and your debugging will need source tracing.

---

## 7. Hands-On Part 1: Split a Document and Explore

Let's use `RecursiveCharacterTextSplitter` on our NLP article  
and really dig into the results.

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader


def split_and_explore(file_path, chunk_size, chunk_overlap):
    """Load a file, split it, and print a detailed analysis."""

    # Step 1: Load the document
    loader = TextLoader(file_path=file_path, encoding="utf-8")
    docs = loader.load()
    original_length = len(docs[0].page_content)

    # Step 2: Create the splitter and split
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)

    # Step 3: Calculate statistics using a list comprehension
    # [len(c.page_content) for c in chunks] creates a list of all chunk lengths
    # e.g., [987, 1000, 856, 923, ...] — one number per chunk
    chunk_lengths = [len(c.page_content) for c in chunks]
    avg_length = sum(chunk_lengths) / len(chunk_lengths)  # average
    min_length = min(chunk_lengths)  # smallest chunk
    max_length = max(chunk_lengths)  # largest chunk

    # Step 4: Print the analysis
    print(f"SPLITTING ANALYSIS")
    print(f"{'=' * 60}")
    print(f"Settings: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
    # :, adds commas to numbers (e.g., 7,234 instead of 7234)
    print(f"Original: {original_length:,} characters")
    print(f"Result: {len(chunks)} chunks")
    print(f"")
    # :,.0f formats as: commas + no decimal places
    print(f"Chunk size stats:")
    print(f"  Average: {avg_length:,.0f} characters")
    print(f"  Smallest: {min_length:,} characters")
    print(f"  Largest: {max_length:,} characters")
    print(f"")

    # Step 5: Show the first 3 chunks (full content — chunks are already small)
    # min(3, len(chunks)) prevents errors if fewer than 3 chunks exist
    for i in range(min(3, len(chunks))):
        print(f"\n--- Chunk {i + 1} ({chunk_lengths[i]} chars) ---")
        print(chunks[i].page_content)  # full chunk content, no slicing needed
        print(f"--- End of chunk {i + 1} ---")

    # Return chunks so they can be used downstream
    return chunks


# Run it!
my_chunks = split_and_explore(
    file_path="data/nlp_article.txt",
    chunk_size=1000,
    chunk_overlap=200,
)

SPLITTING ANALYSIS
Settings: chunk_size=1000, chunk_overlap=200
Original: 8,039 characters
Result: 10 chunks

Chunk size stats:
  Average: 810 characters
  Smallest: 619 characters
  Largest: 992 characters


--- Chunk 1 (773 chars) ---
Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.

NLP combines computational linguistics — rule-based modeling of human language — with statistical, machine learning, and deep learning models. Together, these technologies enable computers to process human language in the form of text or voice data and to understand its full meaning, complete with the speaker's or writer's intent and 

#### What just happened?

We built a reusable `split_and_explore()` function that:
1. Loads any text file
2. Splits it with your chosen settings
3. Shows you stats (count, average/min/max chunk size)
4. Previews the first 3 chunks so you can judge quality

**Look at the chunks:** Do they start and end at natural points?  
Does each chunk contain a complete idea? That's how you evaluate chunking quality.

---

## 8. Hands-On Part 2: The Chunk Size Showdown

Let's split the **same document** with 3 different chunk sizes  
and see how the results compare. This is the most important experiment  
you can run when tuning a RAG system!

In [26]:
# Load the document once
loader = TextLoader(file_path="data/nlp_article.txt", encoding="utf-8")
docs = loader.load()

# Define 3 different chunk sizes to compare
# Each uses 20% overlap (the sweet spot we discussed)
# This is a list of dictionaries — each dict holds one configuration
configurations = [
    {"chunk_size": 300, "chunk_overlap": 60, "label": "Small (300)"},
    {"chunk_size": 1000, "chunk_overlap": 200, "label": "Medium (1000)"},
    {"chunk_size": 2000, "chunk_overlap": 400, "label": "Large (2000)"},
]

# Store results for comparison — empty dict that we'll fill in the loop
results = {}

# This loop runs 3 times — once for each configuration in the list
for config in configurations:
    # config is a dictionary, so config["chunk_size"] gets the value
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )
    chunks = splitter.split_documents(docs)

    # List comprehension: creates a list of chunk lengths in one line
    # Same as writing a for loop to append len(c.page_content) to a list
    chunk_lengths = [len(c.page_content) for c in chunks]

    # Store the results using the label as the dictionary key
    results[config["label"]] = {
        "chunks": chunks,
        "count": len(chunks),
        "avg_length": sum(chunk_lengths) / len(chunk_lengths),
    }

In [27]:
# Print the comparison table
print("CHUNK SIZE SHOWDOWN")
print("=" * 60)

# :<20, :>10, :>15 are format specifiers for column alignment
print(f"\n{'Config':<20} {'Chunks':>10} {'Avg Length':>15}")
print(f"{'-' * 20} {'-' * 10} {'-' * 15}")

# .items() returns each key-value pair from the results dictionary
# label = "Small (300)", data = {"chunks": [...], "count": 25, "avg_length": 280}
for label, data in results.items():
    print(f"{label:<20} {data['count']:>10} {data['avg_length']:>13,.0f}ch")

# Now show the FIRST CHUNK from each configuration
# This is where you really see the difference!
for label, data in results.items():
    # data["chunks"][0] gets the first chunk from this configuration
    # .page_content gets the actual text of that chunk
    first_chunk = data["chunks"][0].page_content
    print(f"\n{'=' * 60}")
    print(f"FIRST CHUNK — {label}")
    print(f"{'=' * 60}")
    print(first_chunk)  # full chunk content — no slicing needed
    print(f"\n[{len(first_chunk)} characters]")

CHUNK SIZE SHOWDOWN

Config                   Chunks      Avg Length
-------------------- ---------- ---------------
Small (300)                  45           201ch
Medium (1000)                10           810ch
Large (2000)                  5         1,698ch

FIRST CHUNK — Small (300)
Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

[99 characters]

FIRST CHUNK — Medium (1000)
Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.

NLP combines computational linguistics — rule-based modeling of human language — with statistical, machine learning, and deep learning mode

#### What just happened?

We split the same document 3 ways and compared the results:

| Config | Chunks | What You See |
|--------|--------|--------------|
| **Small (300)** | Many chunks | Each chunk is a short paragraph. Very precise for search, but might lack context. |
| **Medium (1000)** | Moderate | Each chunk is 1-2 paragraphs. Good balance of precision and context. |
| **Large (2000)** | Few chunks | Each chunk is a full section. Lots of context, but search might return too much noise. |

### Which size should YOU pick?

| Use Case | Recommended Size | Why |
|----------|-----------------|-----|
| FAQ / Q&A bot | 300-500 | Short, focused answers |
| General RAG chatbot | 500-1000 | Good all-around balance |
| Research / analysis | 1000-1500 | Needs more context per chunk |
| Legal / compliance | 1500-2000 | Full clauses need to stay together |

**There is no single right answer.** The best chunk size depends on your data  
and use case. Always experiment with 2-3 sizes and evaluate!

In [28]:
# BONUS: Let's also see how overlap affects the boundary between chunks
# Compare the SAME chunk_size with DIFFERENT overlaps

# A list of 3 configurations — only the overlap value changes
overlap_configs = [
    {"overlap": 0, "label": "No overlap (0)"},
    {"overlap": 100, "label": "Small overlap (100)"},
    {"overlap": 300, "label": "Large overlap (300)"},
]

print("OVERLAP COMPARISON (chunk_size=1000)")
print("=" * 60)

# This loop runs 3 times — once per overlap config
for config in overlap_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=config["overlap"],
    )
    chunks = splitter.split_documents(docs)

    # sum() with a generator expression adds up all chunk content lengths
    # This tells us the total characters stored (including duplicated overlap text)
    total_stored = sum(len(c.page_content) for c in chunks)
    original_length = len(docs[0].page_content)

    # Calculate storage overhead as a percentage
    # overhead = how much EXTRA storage the overlap adds compared to the original
    # e.g., if original is 7000 chars and we store 8400, overhead is +20%
    overhead = ((total_stored / original_length) - 1) * 100

    print(f"\n{config['label']}:")
    print(f"  Chunks: {len(chunks)}")
    # :, adds commas to large numbers for readability
    print(f"  Total chars stored: {total_stored:,} (original: {original_length:,})")
    # :.1f formats the percentage with 1 decimal place
    print(f"  Storage overhead: +{overhead:.1f}%")

OVERLAP COMPARISON (chunk_size=1000)

No overlap (0):
  Chunks: 10
  Total chars stored: 8,020 (original: 8,039)
  Storage overhead: +-0.2%

Small overlap (100):
  Chunks: 10
  Total chars stored: 8,103 (original: 8,039)
  Storage overhead: +0.8%

Large overlap (300):
  Chunks: 10
  Total chars stored: 8,401 (original: 8,039)
  Storage overhead: +4.5%


#### What just happened?

- **No overlap (0):** Fewest chunks, no duplicate text, but context can get lost at boundaries
- **Small overlap (100):** Slightly more chunks, ~10% extra storage, good safety net
- **Large overlap (300):** Most chunks, 30%+ extra storage, maximum boundary safety

**The trade-off is clear:**  
More overlap = better boundary coverage but more storage and more duplicate text in search results.

**The sweet spot (10-20% overlap) gives you boundary safety without blowing up your storage.**

---

## 9. Mini Challenges

### Challenge 1: The Custom Splitter
Split the `data/nlp_article.txt` file with chunk_size=500 and chunk_overlap=75.  
Print how many chunks were created and show the first 2.

### Challenge 2: The Boundary Inspector
Split a document with overlap=100. Then write code to find and print the  
**overlapping text** between chunk 1 and chunk 2. (Hint: compare the end  
of chunk 1 with the start of chunk 2.)

### Challenge 3: The Full Pipeline
Combine what you learned in Lectures 5 and 6:  
1. Load the sample PDF with `PyPDFLoader`  
2. Split it with `RecursiveCharacterTextSplitter`  
3. Add `chunk_index` to each chunk's metadata  
4. Print a summary showing: chunk number, page, and first 50 characters

> **Hint for Challenge 3:** This is the start of a real RAG pipeline!  
> Load → Split → (Next lecture: Embed → Store → Retrieve)

In [20]:
# ============================================================
# Challenge 1: The Custom Splitter
# ============================================================
# Split data/nlp_article.txt with chunk_size=500, chunk_overlap=75
# Print chunk count and the first 2 chunks

# Your code here:


In [21]:
# ============================================================
# Challenge 2: The Boundary Inspector
# ============================================================
# Split with overlap=100, then find the overlapping text
# between chunk 1 and chunk 2.
#
# Hint: end_of_chunk_1 = chunks[0].page_content[-100:]
#        start_of_chunk_2 = chunks[1].page_content[:100]

# Your code here:


In [22]:
# ============================================================
# Challenge 3: The Full Pipeline
# ============================================================
# 1. Load data/sample.pdf with PyPDFLoader
# 2. Split with RecursiveCharacterTextSplitter
# 3. Add chunk_index to each chunk's metadata
# 4. Print: chunk number, page number, first 50 chars
#
# Expected output format:
#   Chunk 1 | Page 0 | "Introduction to NLP Hope to Skill - NLP Co..."
#   Chunk 2 | Page 0 | "Topics covered in this course: 1. Text Pro..."

# Your code here:


---

## 10. Quick Reference: Choosing the Right Splitter & Settings

### Step 1: Pick Your Splitter

| Content Type | Splitter | Why |
|-------------|---------|-----|
| General text | `RecursiveCharacterTextSplitter` | Smart paragraph → sentence → word fallback |
| Markdown/wiki | `MarkdownTextSplitter` | Respects heading structure |
| Source code | `RecursiveCharacterTextSplitter.from_language()` | Respects function/class boundaries |
| Token-sensitive | `TokenTextSplitter` | Exact token counts for API cost control |
| HTML pages | `HTMLSectionSplitter` | Splits on HTML tags |
| Simple/fast | `CharacterTextSplitter` | One separator, maximum speed |

### Step 2: Choose Your Settings

| Use Case | chunk_size | chunk_overlap | Why |
|----------|-----------|--------------|-----|
| FAQ bot | 300-500 | 30-50 | Short, focused answers |
| General chatbot | 500-1000 | 100-200 | Balanced precision + context |
| Research tool | 1000-1500 | 150-300 | Detailed, context-rich chunks |
| Legal / compliance | 1500-2000 | 200-400 | Full clauses stay together |

### Step 3: Always Verify

```python
# The 3-check verification routine after every split:
# 1. Check chunk count — too many or too few?
print(f"Chunks: {len(chunks)}")

# 2. Check chunk sizes — are they consistent?
sizes = [len(c.page_content) for c in chunks]
print(f"Avg: {sum(sizes)/len(sizes):.0f}, Min: {min(sizes)}, Max: {max(sizes)}")

# 3. Read a few chunks — do they make sense on their own?
print(chunks[0].page_content)
```

---

## 11. Key Takeaways

1. **Chunking is critical** — garbage chunks = garbage RAG, no matter how good the LLM
2. **Default to `RecursiveCharacterTextSplitter`** with 500-1500 chars and 10-20% overlap
3. **Use specialized splitters** for markdown, code, and token-sensitive tasks
4. **Always preserve metadata** — your users need citations, you need debugging
5. **Experiment with chunk sizes** — there's no universal "best" size, it depends on your data
6. **The pipeline so far:**
   ```
   Load (Lecture 5) → Split (Lecture 6) → Embed → Store → Retrieve (coming next!)
   ```

### Next Lecture

**Lecture 7: Embeddings & Vector Stores** — Your chunks are ready.  
Now let's convert them into searchable vectors so your AI can find  
the right chunk when a user asks a question!

---

*Hope to Skill — Building the future, one skill at a time.*

---

## Appendix: PEP 8 Style Rules Used in This Notebook

All code in this notebook follows Python's PEP 8 style guide.  
Here are the rules applied, with examples from the code above.

### Naming Conventions

| Rule | Convention | Example from This Notebook |
|------|-----------|---------------------------|
| Variables & functions | `snake_case` | `chunk_size`, `split_and_explore()`, `pdf_chunks` |
| Constants | `UPPER_CASE` | None used in this notebook (configs are variable) |
| Classes | `PascalCase` | `RecursiveCharacterTextSplitter`, `Document` (from LangChain) |

### Import Rules

| Rule ID | Rule | Example |
|---------|------|---------|
| E401 | One import per line | `from langchain_text_splitters import RecursiveCharacterTextSplitter` |
| E402 | Imports at the top of each section | All imports appear at the top of their code cell |

### Whitespace & Formatting

| Rule ID | Rule | Example |
|---------|------|---------|
| E225 | Spaces around operators | `chunk_size - overlap`, `index + 1` |
| E231 | Space after commas | `min(3, len(chunks))` |
| E265 | Block comments start with `# ` | `# Load the document` |
| E303 | Two blank lines before function definitions | See `split_and_explore()` |
| E501 | Max line length of 79 characters | Long strings use slicing or wrapping |

### Best Practices

| Practice | Why | Example |
|----------|-----|---------|
| f-strings for formatting | Cleaner than `.format()` | `f"Chunks: {len(chunks)}"` |
| `enumerate()` for loops | Avoids manual counter variables | `for i, chunk in enumerate(chunks)` |
| List comprehensions | Concise, Pythonic | `[len(c.page_content) for c in chunks]` |
| `min()` for safe limits | Prevents IndexError | `min(3, len(chunks))` |
| Docstrings for functions | Explains the purpose | `split_and_explore()` has a docstring |
| Trailing commas | Cleaner git diffs | All multi-line dicts and function calls |
| Descriptive names | Code reads like English | `chunk_lengths` not `cl`, `pdf_chunks` not `pc` |

### Quick PEP 8 Cheat Sheet

```python
# GOOD (PEP 8 compliant)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(documents)
chunk_lengths = [len(c.page_content) for c in chunks]

# BAD (violates PEP 8)
s = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)  # no space after comma
c = s.split_documents(documents)                                       # single-letter vars
cl = [len(x.page_content) for x in c]                                  # unclear names
```